# Data Description
- Background: 都市發展局提供之都市計畫分區圖資。原始名稱為細計-面與細計-文；後續可能支援 Dataset B2，尚非可分析分區面積。
- date received: 未知；本機下載／收到日期待學生或原始下載紀錄補證，不以文件建立日代填。
- Path to data file: [臺北市都市計畫使用分區圖官方資料集](https://data.taipei/dataset/detail?id=3bab0a01-7936-4218-8cb5-f74dfcb43dda)；來源網址由學生提供，2026-11-25 唯讀查閱詮釋資料，未重新下載原始資料。PDF 下載來源與本機位元組關係未認證。
- Unit of observation: 面圖層為原始分區或範圍幾何紀錄，含 Polygon／MultiPolygon；文圖層為 3 個 Point，標註用途尚待來源規格確認。
- Sample period: 本機版本與圖資適用時間未知；官方目前描述更新至民國 114 年 1 月，不能直接套為本機涵蓋時間或兩年獨立環境資料。
- Known issues: 兩圖層缺 PRJ／CPG、CRS=None；CP950 可讀不代表 CRS 或版本已認證。143 筆無效面幾何、5 組 11 列完全相同幾何、非一般分區及部分重疊待處理。
- Definition for each variable: 見 Summary Stats 的完整逐欄定義表；「觀測內容」與「官方定義」分列，官方未定義處明記未知。
- Data path: [細計-面.shp](細計-面.shp)、[細計-面.shx](細計-面.shx)、[細計-面.dbf](細計-面.dbf)、[細計-文.shp](細計-文.shp)、[細計-文.shx](細計-文.shx)、[細計-文.dbf](細計-文.dbf)；程式讀檔使用下方精確的工作包相對路徑。
- Role: `raw`（參考 PDF 亦逐檔登記）
- Verification status: 已實際執行唯讀文件化檢查；詳見保存輸出。檔案身分、基本結構及列明診斷可核對，不認證為 analysis-ready。
- SHA-256: 見下方固定綁定值及 Summary 逐檔大小／hash 表；每次讀檔前後重新計算。
- Next processing notebook: pending；尚未建立，不預設處理檔名。
- Author / contributors: 詹明翰、謝子詮、曾翊涵（學生確認三人共同完成本次 Unit 0.5；不推定個別分工）
- Created At: 2026-11-25（文件建立日，與 date received 不同）

**官方版本資訊與本機版本分開記錄：** 官方提供機關為都發局，頁面描述主要計畫暨細部計畫分區圖、更新至 114 年 1 月；細計資源更新 2026-04-09 11:49:14、詮釋資料更新 11:49:40。本次僅查閱頁面，未下載原包；本機版本及缺少配套原因未知。

本文件依 `templates/data/raw/template-doc-and-summary-stats.ipynb` 的原欄位與章節順序建立。
僅描述未修改原始資料；未刪除、補值、修復、正式轉換或輸出 processed。正式研究與規劃定義詳見 [PROJECT.md](../../PROJECT.md)。


In [1]:
from pathlib import Path
import hashlib, json, sys, platform, os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('Open within the project containing AGENTS.md and templates/.')

PROJECT_ROOT = find_project_root()
assert Path(sys.executable).resolve() == (PROJECT_ROOT / '.venv/Scripts/python.exe').resolve()
assert sys.prefix != sys.base_prefix
TEMP = PROJECT_ROOT / 'data/temp/unit05_documentation'
for variable, folder in {'MPLCONFIGDIR':'matplotlib', 'IPYTHONDIR':'ipython',
                         'JUPYTER_RUNTIME_DIR':'runtime'}.items():
    location = TEMP / folder
    location.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(location)
os.environ['PROJ_NETWORK'] = 'OFF'
pd.set_option('display.max_colwidth', 100)

def project_file(relative_path, area='raw'):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('Explicit project-relative path required')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('Input path outside permitted data area')
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify_inputs(expected):
    rows = []
    for relative_path, digest in expected.items():
        path = project_file(relative_path)
        actual = sha256_file(path)
        if actual != digest:
            raise ValueError(f'Input changed; review provenance before loading: {relative_path}')
        rows.append({'Data path':relative_path, 'bytes':path.stat().st_size,
                     'format':path.suffix.lower(), 'SHA-256':actual})
    return pd.DataFrame(rows)

def show_table(title, frame, block=25):
    display(Markdown('**' + title + '**'))
    if not len(frame):
        display(Markdown('無符合紀錄。'))
    else:
        for start in range(0, len(frame), block):
            display(frame.iloc[start:start + block])

def sample_ten(title, frame, block_columns=12):
    display(Markdown('**' + title + '：實際前十列，按欄分組呈現；未抽樣或排序**'))
    for start in range(0, len(frame.columns), block_columns):
        display(frame.iloc[:10, start:start + block_columns])

def schema(frame, definitions, levels=None):
    rows=[]
    for column in frame.columns:
        meaning, evidence = definitions[column]
        rows.append({'欄位':column, '讀入型態':str(frame[column].dtype),
                     '定義或可觀察內容':meaning, '定義依據及限制':evidence,
                     '層級':levels.get(column, '原始紀錄') if levels else '原始紀錄',
                     '缺失數':int(frame[column].isna().sum()),
                     '非缺失相異值':int(frame[column].nunique(dropna=True))})
    return pd.DataFrame(rows)

def finish_check():
    for relative_path,digest in EXPECTED_SHA256.items():
        assert sha256_file(project_file(relative_path)) == digest, relative_path
    print({'input_hashes_unchanged':True, 'input_files_checked':len(EXPECTED_SHA256),
           'interpreter_matches_project_venv':True, 'python_version':platform.python_version(),
           'scope':'read-only raw documentation; no processed dataset or model output'})


## Change for Your Own Data
以下固定原始檔路徑與事前檢查所得 SHA-256；版本不同即停止，不換用其他檔案。
CSV 的編碼／代碼型態、Shapefile 的 reader 與配套檔均明列。讀入型態是本次 reader 的推斷或明示設定，不冒稱官方儲存型態。


In [2]:
EXPECTED_SHA256 = {'data/raw/細計-面.shp': '67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee', 'data/raw/細計-面.shx': '277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69', 'data/raw/細計-面.dbf': '29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4', 'data/raw/細計-文.shp': 'fa1767fc78c69b7683ec53e0fab8937bac6c33868bedee3c3dcc2ed38dcab731', 'data/raw/細計-文.shx': '32d642964d494f49d08830f9c12857f9f4995b56eebf09ef2be994034b864760', 'data/raw/細計-文.dbf': 'eba92881fc11e574f24024380c946d73cf213e86040663c4070d15b1d552a7be'}
INPUT_MANIFEST = verify_inputs(EXPECTED_SHA256)


## Summary


In [3]:
show_table('表 1：原始輸入逐檔檔案格式、大小及 SHA-256', INPUT_MANIFEST)
import geopandas as gpd
layers={label:gpd.read_file(project_file(path),engine='pyogrio',encoding='cp950') for label,path in {'面':'data/raw/細計-面.shp','文':'data/raw/細計-文.shp'}.items()}
show_table('表 2：原始圖層結構',pd.DataFrame([{'圖層':label,'列數':len(f),'屬性欄':len(f.columns)-1,'含geometry欄數':len(f.columns),'CRS':str(f.crs),'文字編碼':'CP950（讀取可行，缺CPG）'} for label,f in layers.items()]))


**表 1：原始輸入逐檔檔案格式、大小及 SHA-256**

,Data path,bytes,format,SHA-256
0,data/raw/細計-面.shp,7556848,.shp,67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee
1,data/raw/細計-面.shx,124244,.shx,277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69
2,data/raw/細計-面.dbf,13982136,.dbf,29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4
3,data/raw/細計-文.shp,184,.shp,fa1767fc78c69b7683ec53e0fab8937bac6c33868bedee3c3dcc2ed38dcab731
4,data/raw/細計-文.shx,124,.shx,32d642964d494f49d08830f9c12857f9f4995b56eebf09ef2be994034b864760
5,data/raw/細計-文.dbf,3121,.dbf,eba92881fc11e574f24024380c946d73cf213e86040663c4070d15b1d552a7be


**表 2：原始圖層結構**

,圖層,列數,屬性欄,含geometry欄數,CRS,文字編碼
0,面,15518,12,13,None,CP950（讀取可行，缺CPG）
1,文,3,12,13,None,CP950（讀取可行，缺CPG）


### Sample


In [4]:
for label,frame in layers.items():
    sample_ten(f'表 3：{label}圖層原始樣本',frame)


**表 3：面圖層原始樣本：實際前十列，按欄分組呈現；未抽樣或排序**

,編號,圖層,顏色,街廓編號,分區代碼,分區簡稱,使用分區,分區說明,原屬分區,變更前代碼,變更前簡稱,變更前分區
0,1,368,0,NaN,細部計畫範圍,細部計畫範圍,NaN,NaN,NaN,None,None,None
1,2,370,0,NaN,規劃範圍,規劃範圍,NaN,NaN,NaN,None,None,None
2,3,370,0,NaN,規劃範圍,規劃範圍,NaN,NaN,NaN,None,None,None
3,4,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None
4,5,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None
5,6,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None
6,7,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None
7,8,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None
8,9,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None
9,10,32,19,NaN,PEA,公,公園用地,NaN,NaN,None,None,None


,geometry
0,"POLYGON ((294770.795 2790592.095, 319674.63 2790592.095, 319674.63 2759513.054, 294770.795 27595..."
1,"POLYGON ((294770.795 2790592.095, 319674.63 2790592.095, 319674.63 2759513.054, 294770.795 27595..."
2,"POLYGON ((294770.795 2790592.095, 319674.63 2790592.095, 319674.63 2759513.054, 294770.795 27595..."
3,"POLYGON ((303340.099 2771175.776, 303342.974 2771175.001, 303347.478 2771179.356, 303361.271 277..."
4,"POLYGON ((303230.925 2771108.301, 303230.925 2771108.3, 303242.707 2771104.354, 303247.503 27711..."
5,"POLYGON ((303073.502 2771160.531, 303076.954 2771162.553, 303081.636 2771180.618, 303079.603 277..."
6,"POLYGON ((302981.707 2771184.274, 302985.146 2771186.317, 302988.43 2771196.018, 302989.843 2771..."
7,"POLYGON ((302883.036 2771199.257, 302886.639 2771201.92, 302890.32 2771216.57, 302890.315 277121..."
8,"POLYGON ((302821.988 2771202.441, 302825.575 2771205.101, 302829.005 2771219.083, 302736.257 277..."
9,"POLYGON ((302704.335 2771209.099, 302708.422 2771211.979, 302712.451 2771227.371, 302710.335 277..."


**表 3：文圖層原始樣本：實際前十列，按欄分組呈現；未抽樣或排序**

,編號,圖層,顏色,街廓編號,分區代碼,分區簡稱,使用分區,分區說明,原屬分區,變更前代碼,變更前簡稱,變更前分區
0,1,64,253,None,CIA7,NaN,None,None,None,None,None,None
1,2,72,253,None,PEA,公,None,None,None,None,None,None
2,3,72,253,None,PEA,公,None,None,None,None,None,None


,geometry
0,POINT (305939.493 2783895.697)
1,POINT (305233.534 2765289.193)
2,POINT (305227.027 2765292.433)


### Summary Stats


In [5]:
DEFINITIONS = {'編號': ('本機圖層內編號字串；是否為跨版本穩定 ID 未知', '官方列欄名；穩定性及產製規格未取得'), '圖層': ('圖層類別代碼字串；列明實際分布，不擅自解釋每個碼', '官方完整圖層碼表未取得'), '顏色': ('原始顏色代碼字串；顏色系統及語意未知', '官方僅列欄名'), '街廓編號': ('原始街廓編號欄；多數空白，正式編碼規則未知', '官方僅列欄名'), '分區代碼': ('本機分區代碼或範圍標籤；同時有非一般分區文字', '官方僅列欄名；原始代碼分布及關係表見輸出'), '分區簡稱': ('本機分區簡稱文字；不由簡稱直接建立研究分類', '官方僅列欄名；本次顯示實際對應'), '使用分區': ('本機分區名稱／用途標籤；不是實際土地使用調查', '官方僅列欄名；有空白，不能視為 0 或自動分類'), '分區說明': ('本機補充文字欄；多數空白，與其他欄關係以輸出為準', '官方僅列欄名；法律／分類解讀待正式說明'), '原屬分區': ('原始原屬分區文字欄；時間與變更規格未知', '官方僅列欄名'), '變更前代碼': ('原始變更前代碼欄；本機全缺失，不推定未曾變更', '官方僅列欄名；版本、缺值語意未知'), '變更前簡稱': ('原始變更前簡稱欄；本機全缺失', '官方僅列欄名；時間與產製規則未知'), '變更前分區': ('原始變更前分區欄；本機全缺失', '官方僅列欄名；時間與產製規則未知'), 'geometry': ('GeoPandas 讀入的面／多面或點幾何，CRS=None', '實際 SHP 讀取；不是 DBF 屬性欄，不自行指派 CRS')}
for layer,frame in layers.items():
    show_table(f'表 4：{layer} 全 12 屬性欄＋geometry 定義及缺失',schema(frame,DEFINITIONS),block=15)
    show_table(f'表 5：{layer} 原始屬性描述統計',frame.drop(columns='geometry').describe(include='all').T)


**表 4：面 全 12 屬性欄＋geometry 定義及缺失**

,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
0,編號,str,本機圖層內編號字串；是否為跨版本穩定 ID 未知,官方列欄名；穩定性及產製規格未取得,原始紀錄,0,15518
1,圖層,str,圖層類別代碼字串；列明實際分布，不擅自解釋每個碼,官方完整圖層碼表未取得,原始紀錄,0,97
2,顏色,str,原始顏色代碼字串；顏色系統及語意未知,官方僅列欄名,原始紀錄,0,20
3,街廓編號,str,原始街廓編號欄；多數空白，正式編碼規則未知,官方僅列欄名,原始紀錄,15505,13
4,分區代碼,str,本機分區代碼或範圍標籤；同時有非一般分區文字,官方僅列欄名；原始代碼分布及關係表見輸出,原始紀錄,0,260
5,分區簡稱,str,本機分區簡稱文字；不由簡稱直接建立研究分類,官方僅列欄名；本次顯示實際對應,原始紀錄,0,241
6,使用分區,str,本機分區名稱／用途標籤；不是實際土地使用調查,官方僅列欄名；有空白，不能視為 0 或自動分類,原始紀錄,4,253
7,分區說明,str,本機補充文字欄；多數空白，與其他欄關係以輸出為準,官方僅列欄名；法律／分類解讀待正式說明,原始紀錄,15409,34
8,原屬分區,str,原始原屬分區文字欄；時間與變更規格未知,官方僅列欄名,原始紀錄,15224,13
9,變更前代碼,object,原始變更前代碼欄；本機全缺失，不推定未曾變更,官方僅列欄名；版本、缺值語意未知,原始紀錄,15518,0


**表 5：面 原始屬性描述統計**

,count,unique,top,freq
編號,15518,15518,1,1
圖層,15518,97,5,5122
顏色,15518,20,4,7949
街廓編號,13,13,C1,1
分區代碼,15518,260,R3,4910
分區簡稱,15518,241,住3,4910
使用分區,15514,253,第三種住宅區,4910
分區說明,109,34,(二),20
原屬分區,294,13,住3,176
變更前代碼,0,0,NaN,NaN


**表 4：文 全 12 屬性欄＋geometry 定義及缺失**

,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
0,編號,str,本機圖層內編號字串；是否為跨版本穩定 ID 未知,官方列欄名；穩定性及產製規格未取得,原始紀錄,0,3
1,圖層,str,圖層類別代碼字串；列明實際分布，不擅自解釋每個碼,官方完整圖層碼表未取得,原始紀錄,0,2
2,顏色,str,原始顏色代碼字串；顏色系統及語意未知,官方僅列欄名,原始紀錄,0,1
3,街廓編號,object,原始街廓編號欄；多數空白，正式編碼規則未知,官方僅列欄名,原始紀錄,3,0
4,分區代碼,str,本機分區代碼或範圍標籤；同時有非一般分區文字,官方僅列欄名；原始代碼分布及關係表見輸出,原始紀錄,0,2
5,分區簡稱,str,本機分區簡稱文字；不由簡稱直接建立研究分類,官方僅列欄名；本次顯示實際對應,原始紀錄,1,1
6,使用分區,object,本機分區名稱／用途標籤；不是實際土地使用調查,官方僅列欄名；有空白，不能視為 0 或自動分類,原始紀錄,3,0
7,分區說明,object,本機補充文字欄；多數空白，與其他欄關係以輸出為準,官方僅列欄名；法律／分類解讀待正式說明,原始紀錄,3,0
8,原屬分區,object,原始原屬分區文字欄；時間與變更規格未知,官方僅列欄名,原始紀錄,3,0
9,變更前代碼,object,原始變更前代碼欄；本機全缺失，不推定未曾變更,官方僅列欄名；版本、缺值語意未知,原始紀錄,3,0


**表 5：文 原始屬性描述統計**

,count,unique,top,freq
編號,3,3,1,1
圖層,3,2,72,2
顏色,3,1,253,3
街廓編號,0,0,NaN,NaN
分區代碼,3,2,PEA,2
分區簡稱,2,1,公,2
使用分區,0,0,NaN,NaN
分區說明,0,0,NaN,NaN
原屬分區,0,0,NaN,NaN
變更前代碼,0,0,NaN,NaN


#### 配套、分區內容、重複與幾何診斷
兩圖層均有 SHP（幾何）、SHX（索引）、DBF（屬性），缺 PRJ 與 CPG；各現有檔案均已逐檔 hash 綁定。
完全相同幾何以原始 WKB 位元組相同判定；不等於已檢查所有拓樸相等、方向不同或部分重疊圖形。
有效性檢查在原始平面數值中進行，不為此指定 CRS、不算面積，也不與里界直接疊合。


In [6]:
import shapely
quality=[]
for layer,frame in layers.items():
    stem='細計-面' if layer=='面' else '細計-文'
    missing=[extension for extension in ['prj','cpg'] if not (PROJECT_ROOT / f'data/raw/{stem}.{extension}').exists()]
    wkb=frame.geometry.to_wkb()
    repetitions=wkb.value_counts()
    repeated=repetitions.loc[repetitions.gt(1)]
    quality.append({'圖層':layer,'筆數':len(frame),'CRS':str(frame.crs),'缺少配套':','.join(missing),
       '空白幾何':int(frame.geometry.is_empty.sum()),'缺失幾何':int(frame.geometry.isna().sum()),
       '無效幾何':int((~frame.geometry.is_valid).sum()),'完全相同WKB群組':len(repeated),
       '重複群組涉及列':int(repeated.sum()),'重複群組多餘列數':int((repeated-1).sum()),
       '完整屬性重複列':int(frame.drop(columns='geometry').duplicated().sum())})
    assert frame.crs is None
    show_table(f'表 6：{layer} 幾何類型分布',frame.geom_type.value_counts().rename('筆數').to_frame())
    reasons=pd.Series(shapely.is_valid_reason(frame.geometry.array)).str.split('[').str[0]
    show_table(f'表 7：{layer} 幾何有效性理由；未修復',reasons.value_counts().rename('筆數').to_frame())
    for column in ['圖層','分區代碼']:
        show_table(f'表 8：{layer} {column} 完整原始分布；分母={len(frame):,}',
                   frame[column].value_counts(dropna=False).rename('原始列數').to_frame(),block=30)
    relation=(frame.groupby(['分區代碼','分區簡稱','使用分區','分區說明'],dropna=False)
                    .size().rename('原始列數').reset_index())
    show_table(f'表 9：{layer} 分區代碼／簡稱／使用分區／說明實際關係；未建立分類',relation,block=25)
    blank=frame['使用分區'].isna()|frame['使用分區'].fillna('').str.strip().eq('')
    show_table(f'表 10：{layer} 使用分區空白的全部原始屬性列（非整體資料傾印）',
               frame.loc[blank].drop(columns='geometry'))
    duplicated=frame.loc[wkb.isin(repeated.index)].drop(columns='geometry')
    show_table(f'表 11：{layer} 完全相同幾何涉及的全部屬性；未刪除',duplicated)
    print({'layer':layer,'exact_WKB_duplicate_group_sizes':repeated.to_list(),
           'blank_zoning_rows':int(blank.sum()),'nonidentical_partial_overlaps':'not tested; Chapter 1',
           'unknown_CRS_not_assigned':True,'no_area_calculation':True})
show_table('表 12：圖資品質摘要與缺少配套',pd.DataFrame(quality))


**表 6：面 幾何類型分布**

,筆數
Polygon,15508
MultiPolygon,10


**表 7：面 幾何有效性理由；未修復**

,筆數
Valid Geometry,15375
Self-intersection,67
Ring Self-intersection,52
Hole lies outside shell,15
Nested shells,9


**表 8：面 圖層 完整原始分布；分母=15,518**

,原始列數
圖層,
5,5122
17,1456
32,1286
2,653
6,519
42,479
7,469
16,465
10,448


... (44 outputs left out)

無符合紀錄。

{'layer': '文', 'exact_WKB_duplicate_group_sizes': [], 'blank_zoning_rows': 3, 'nonidentical_partial_overlaps': 'not tested; Chapter 1', 'unknown_CRS_not_assigned': True, 'no_area_calculation': True}


**表 12：圖資品質摘要與缺少配套**

,圖層,筆數,CRS,缺少配套,空白幾何,缺失幾何,無效幾何,完全相同WKB群組,重複群組涉及列,重複群組多餘列數,完整屬性重複列
0,面,15518,None,"prj,cpg",0,0,143,5,11,6,0
1,文,3,None,"prj,cpg",0,0,0,0,0,0,0


#### 與 Unit 0.2 的差異及後續條件
總列數 15,518／3、CP950 可讀、缺 PRJ／CPG、CRS=None、5 組共 11 列完全相同幾何均一致。
先前「15,518 筆 Polygon」是面類圖形的概括，本次精確辨識為 15,508 Polygon＋10 MultiPolygon；
新增確認 143 筆無效幾何：67 自交、52 環自交、15 孔洞位於外殼外、9 巢狀外殼。
使用分區空白的 4 筆面紀錄是細部計畫範圍、規劃範圍兩筆及陽明山公園；不刪除、不補成未分類零值。
重複群組涉及 11 列不等於有 11 列多餘；5 組共多出 6 列，但本次全部保留。

後續先補原始下載包、CRS 及版本證據，再審查圖層用途、正式分區分類、重複／重疊及無效幾何的處理規則。
來源驗證與分類確認後才能計算不重複的分區面積及各里比例；分區未覆蓋或缺失不能當 0。
都市計畫分區不等於實際土地使用。此處完成的是文件化，不宣稱已完成正式圖資驗證。


## Cross-Dataset Documentation — 僅規劃、未正式整合

| 原始資料 | 原始觀察單位／時間／空間粒度 | 預計連接與可支持變數 | 整合前條件 |
| --- | --- | --- | --- |
| 2024、2025 事故 | 2024 為當事人紀錄；2025 僅第一當事人；事故時分與點位 | 第一當事人候選事件；驗證 CRS 後以點在行政里面歸里；A 事故組成、C 負擔 | 官方事件口徑、照明欄位跨年映射、缺失／異常及座標基準；邊界點／未匹配規則另審查。 |
| 人口 | 年×月×區域×性別，含市／區／里 | 里層級「計」；12 月平均存量、人口結構 B1 | 代碼 crosswalk、糖廍里例外、月資料基準日與里界適用性；不視為交通暴露人口。 |
| 行政里界 | 456 個里面單元、單一未認證年代快照 | CPTVID 候選空間 ID；人口名稱及代碼驗證後連接，並作事故歸里 | 適用年度、幾何重疊／縫隙、原 AREA 定義與重算口徑；不因里數相同認證兩年幾何一致。 |
| 都市計畫 | 細計面與點、未認證年代快照、未知 CRS | 確認後以面圖層分類與疊合支援 B2 分區比例；點圖層用途待確認 | 原包、CRS、版本、143 筆無效幾何、重複及部分重疊、非一般分區、正式分類表。 |

| 規劃資料集 | 來源及變數 | 狀態與解釋限制 |
| --- | --- | --- |
| Dataset A | 事故；早／晚尖峰、深夜、週末、路口、追撞、側撞、人車、多人受傷組成 | 尚未建立；保留分子／有效分母／缺失／涵蓋率，不用總人口作事故組成分母。 |
| Dataset B | 人口 B1＋通過驗證的都市計畫 B2；其他道路環境 B3 尚未取得或選用 | 尚未建立；不納入事故結果，不以事故中的道路情境代表全里環境；分區不是實際土地使用。 |
| Dataset C | 事故數、傷亡、每件受傷數、事故空間密度、低樣本標記 | 尚未建立；24 小時與 2–30 日死亡分列，缺值不補成 0；每平方公里事故數不是暴露校正事故率。 |

village-year 為後續基礎資料結構；兩年比例合併須合計分子／有效分母再重算，
不能直接平均年度比例。零事故里及低樣本門檻仍待審查。本文件未建立 crosswalk、
年度人口表或 A／B／C，亦未執行事故歸里、土地使用疊合、分群或統計推論。

其他原始資料說明：
[事故](taipei_2024_2025_crashes_raw_documentation.ipynb)、[人口](taipei_population_age_raw_documentation.ipynb)、[里界](taipei_village_boundaries_raw_documentation.ipynb)、[都市計畫](taipei_urban_planning_raw_documentation.ipynb).


In [7]:
finish_check()


{'input_hashes_unchanged': True, 'input_files_checked': 6, 'interpreter_matches_project_venv': True, 'python_version': '3.12.2', 'scope': 'read-only raw documentation; no processed dataset or model output'}
